# **PyTorch CNN Architecture Implementation**

This notebook implements the specified Convolutional Neural Network (CNN) architecture in PyTorch and verifies feature map spatial dimensions at each forward pass step.

The spatial output shape $(H_{out}, W_{out})$ for a 2D layer is computed using the formula:$$H_{out} = \left\lfloor \frac{H_{in} - \text{kernel\_size} + 2 \times \text{padding}}{\text{stride}} \right\rfloor + 1$$Input: $(1, 28, 28)$Conv1 + ReLU (in_channels=1, out_channels=32, kernel=3, stride=1, padding=1):$$H_{out} = \left\lfloor \frac{28 - 3 + 2(1)}{1} \right\rfloor + 1 = 28 \implies (32, 28, 28)$$MaxPool1 (kernel=2, stride=2, padding=1):$$H_{out} = \left\lfloor \frac{28 - 2 + 2(1)}{2} \right\rfloor + 1 = \left\lfloor \frac{28}{2} \right\rfloor + 1 = 15 \implies (32, 15, 15)$$Conv2 + ReLU (in_channels=32, out_channels=64, kernel=3, stride=1, padding=1):$$H_{out} = \left\lfloor \frac{15 - 3 + 2(1)}{1} \right\rfloor + 1 = 15 \implies (64, 15, 15)$$Conv3 + ReLU (in_channels=64, out_channels=128, kernel=3, stride=1, padding=1):$$H_{out} = \left\lfloor \frac{15 - 3 + 2(1)}{1} \right\rfloor + 1 = 15 \implies (128, 15, 15)$$Conv4 + ReLU (in_channels=128, out_channels=256, kernel=3, stride=1, padding=1):$$H_{out} = \left\lfloor \frac{15 - 3 + 2(1)}{1} \right\rfloor + 1 = 15 \implies (256, 15, 15)$$MaxPool2 (kernel=2, stride=2, padding=0):$$H_{out} = \left\lfloor \frac{15 - 2 + 2(0)}{2} \right\rfloor + 1 = \left\lfloor \frac{13}{2} \right\rfloor + 1 = 6 + 1 = 7 \implies (256, 7, 7)$$Flatten:$$\text{Total Features} = 256 \times 7 \times 7 = 12,544$$Fully Connected Layers (FCN):FCN1: $12,544 \rightarrow 1,000$FCN2: $1,000 \rightarrow 500$FCN3: $500 \rightarrow \text{num\_classes}$ (e.g., 10)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNArchitecture(nn.Module):
    def __init__(self, num_classes=10):
        super(CNNArchitecture, self).__init__()

        # Convolutional Block 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2, padding=1)

        # Convolutional Block 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        # Fully Connected Layers
        self.fc1 = nn.Linear(in_features=256 * 7 * 7, out_features=1000)
        self.fc2 = nn.Linear(in_features=1000, out_features=500)
        self.fc3 = nn.Linear(in_features=500, out_features=num_classes)

    def forward(self, x, verbose=False):
        if verbose:
            print(f"Input Shape: {x.shape}")

        x = F.relu(self.conv1(x))
        if verbose: print(f"Conv1 + ReLU Output: {x.shape}")

        x = self.pool1(x)
        if verbose: print(f"MaxPool1 Output: {x.shape}")

        x = F.relu(self.conv2(x))
        if verbose: print(f"Conv2 + ReLU Output: {x.shape}")

        x = F.relu(self.conv3(x))
        if verbose: print(f"Conv3 + ReLU Output: {x.shape}")

        x = F.relu(self.conv4(x))
        if verbose: print(f"Conv4 + ReLU Output: {x.shape}")

        x = self.pool2(x)
        if verbose: print(f"MaxPool2 Output: {x.shape}")

        x = torch.flatten(x, start_dim=1)
        if verbose: print(f"Flattened Output: {x.shape}")

        x = F.relu(self.fc1(x))
        if verbose: print(f"FCN1 + ReLU Output: {x.shape}")

        x = F.relu(self.fc2(x))
        if verbose: print(f"FCN2 + ReLU Output: {x.shape}")

        x = self.fc3(x)
        if verbose: print(f"FCN3 Output (Final): {x.shape}")

        return x

# Testing with dummy input matching dimensions (Batch size = 1, Channels = 1, Height = 28, Width = 28)
model = CNNArchitecture(num_classes=10)
dummy_input = torch.randn(1, 1, 28, 28)

print("=== Spatial Dimension Verification ===")
output = model(dummy_input, verbose=True)

=== Spatial Dimension Verification ===
Input Shape: torch.Size([1, 1, 28, 28])
Conv1 + ReLU Output: torch.Size([1, 32, 28, 28])
MaxPool1 Output: torch.Size([1, 32, 15, 15])
Conv2 + ReLU Output: torch.Size([1, 64, 15, 15])
Conv3 + ReLU Output: torch.Size([1, 128, 15, 15])
Conv4 + ReLU Output: torch.Size([1, 256, 15, 15])
MaxPool2 Output: torch.Size([1, 256, 7, 7])
Flattened Output: torch.Size([1, 12544])
FCN1 + ReLU Output: torch.Size([1, 1000])
FCN2 + ReLU Output: torch.Size([1, 500])
FCN3 Output (Final): torch.Size([1, 10])


## Key Takeaways

The complete Convolutional Neural Network (CNN) architecture was defined from initial feature extraction through pooling and linear classification layers. Performing precise step-by-step spatial dimension calculations for each layer ensures full compatibility across tensor shapes, preventing dimensional mismatch errors when transitioning from convolutional feature maps to fully connected input layers.

This architectural framework was subsequently implemented in PyTorch using a custom model class. Layer declarations in the initialization method directly reflect the calculated spatial transitions, providing a modular structure for complex feature processing.

Finally, executing a forward pass with dummy tensor inputs verified the spatial dimensions at each intermediate layer. The empirical output shapes matched the expected theoretical calculations, demonstrating a reliable methodology for debugging network architectures and tracking feature transformations in PyTorch.